# 🎙️ Voice AI Studio Arabic - Google Colab

منصة صوتيات عربية لتوليد واستنساخ الصوت مفتوحة المصدر

## الخطوات
1. شغّل الخلية الأولى لتثبيت المتطلبات
2. شغّل الخلية الثانية لاستنساخ المشروع
3. شغّل الخلية الثالثة لتشغيل الخادم
4. افتح الرابط الذي يظهر

In [ ]:
# Cell 1: Install requirements
!pip install -q fastapi uvicorn pydantic python-multipart httpx aiofiles nest-asyncio
!pip install -q google-genai kokoro TTS soundfile librosa pydub
print('✅ Requirements installed')

In [ ]:
# Cell 2: Clone project and setup
import os
if not os.path.exists('awesome-voice-ai-tools'):
    !git clone https://github.com/Agfsgsy/awesome-voice-ai-tools.git
%cd /content/awesome-voice-ai-tools

# Create directories
for d in ['uploads', 'outputs', 'cache', 'logs', 'models', 'voices', 'downloads', 'config']:
    os.makedirs(d, exist_ok=True)

import sys
sys.path.insert(0, '/content/awesome-voice-ai-tools')
print('✅ Project ready')

In [ ]:
# Cell 3: Set environment variables (optional)
import os
os.environ['GEMINI_API_KEY'] = ''  # Add your key here
os.environ['GEMINI_TTS_MODEL'] = 'gemini-3.1-flash-tts-preview'
os.environ['APP_PORT'] = '8000'
os.environ['APP_HOST'] = '0.0.0.0'
print('✅ Environment configured')

In [ ]:
# Cell 4: Start server with public URL
import nest_asyncio
nest_asyncio.apply()

import threading
import uvicorn
import time

def run_server():
    uvicorn.run('main:app', host='0.0.0.0', port=8000, log_level='info')

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

time.sleep(3)
print('✅ Server started on http://localhost:8000')
print('📋 API docs: http://localhost:8000/docs')

In [ ]:
# Cell 5: Expose via cloudflared (public URL)
import subprocess
import time

# Install cloudflared
subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', '/usr/local/bin/cloudflared'])
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'])

# Start tunnel
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

time.sleep(5)
print('⏳ Waiting for tunnel URL...')

# Read output to find URL
for line in proc.stdout:
    line = line.decode('utf-8')
    if 'trycloudflare.com' in line:
        print(f'\n🔗 Public URL: {line.strip()}')
        break
    print(line, end='')

## اختبار سريع
استخدم الخلية التالية لاختبار توليد الصوت

In [ ]:
# Cell 6: Quick test
import httpx
import asyncio

async def test_tts():
    async with httpx.AsyncClient() as client:
        # Test health
        r = await client.get('http://localhost:8000/health')
        print(f'Health: {r.status_code}')
        print(r.json().get('status', 'unknown'))

        # Test TTS
        r = await client.post('http://localhost:8000/api/tts', json={
            'text': 'مرحباً بالعالم',
            'engine': 'kokoro',
            'language': 'ar'
        })
        data = r.json()
        print(f'TTS: {r.status_code}')
        print(data.get('message', 'no message'))
        if data.get('url'):
            print(f'Audio URL: {data["url"]}')

asyncio.get_event_loop().run_until_complete(test_tts())